We will directly call data from catalog's bronze schema 
Reading it directly in spark data frame.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType


df_bronze = spark.read.table("weather.bronze.bronze_weather")
display(df_bronze.limit(20))

In [0]:
null_counts = [F.count(F.when(F.col(c).isNull(), c)) for c in df_bronze.columns]
total_nulls = df_bronze.select(sum(null_counts).alias("total_nulls")).collect()[0][0]

display(f"Total null values in df_bronze: {total_nulls}")

In [0]:
# Count original rows
original_count = df_bronze.count()

# Drop duplicates based on the primary key ('time')
df_silver_step1 = df_bronze.dropDuplicates(["time"])

# Count after removing duplicates
deduped_count = df_silver_step1.count()

display(f"Original rows: {original_count}")
display(f"Rows after removing duplicates: {deduped_count}")


In [0]:
# Mapping months to seasons Summer, Winter, Monsoon

df_silver_step2 = df_silver_step1.withColumn(
    "weather",
    F.when(F.month("time").isin(12, 1, 2), "winter")
     .when(F.month("time").isin(3, 4, 5, 6), "summer")
     .otherwise("monsoon")
)

display(df_silver_step2.select("time", "weather").limit(10)) 

In [0]:
# Extract time elements into dedicated columns for downstream filtering and partitioning
df_silver_step3 = df_silver_step2 \
    .withColumn("date", F.to_date("time")) \
    .withColumn("year", F.year("time")) \
    .withColumn("month", F.month("time")) \
    .withColumn("hour", F.hour("time"))

display(df_silver_step3.select("time", "year", "month", "date", "hour", "weather").limit(5))

In [0]:
# 4. Generating the 10-year dynamic partition key (to prevent the Small Files Problem)
df_silver_step4 = df_silver_step3.withColumn(
    "decade_partition", 
    (F.floor(F.col("year") / 5) * 5).cast("int")
)

In [0]:
# Save the processed data as a partitioned Silver Delta table
(df_silver_step4.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("decade_partition")
    .saveAsTable("weather.silver.silver_weather")
)